# 02 — Model check (Qwen2.5-Omni-7B, CPU)

Before any experiment, check that:
1. the pinned model loads;
2. the prompt looks as expected;
3. we get probabilities for the answer letters A–D;
4. one silent video clip with **64 frames** (as in the paper) runs, and how long it takes.

No answer keys are used here.

In [1]:
import sys, time, tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import qwen_omni as qo

CFG = tomllib.loads((ROOT / "configs/2026-09-28_pilot.toml").read_text())
REVISION = CFG["model_revision"]
print(qo.MODEL_ID, "@", REVISION)

Qwen/Qwen2.5-Omni-7B @ ae9e1690543ffd5c0221dc27f79834d0294cba00


## 1. Load the model (~18 GB RAM, a few minutes)

In [2]:
processor, model = qo.load(REVISION)
print("parameters: %.2f billion" % (sum(p.numel() for p in model.parameters()) / 1e9))

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs2.{0, 1, 2}.weight                              | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.weight                                                     | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.bias                                                       | UNEXPECTED |  | 
talker.model.layers.{0...23}.input_layernorm.weight                                                      | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.v_proj.bias                                                       | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.re

parameters: 8.93 billion


## 2. Text-only test

A trivial question where the right answer is **B**. `prob` should favour B, and `mass_on_letters` should be close to 1
(meaning the model really wants to answer with a letter).

In [3]:
SYSTEM = "You are a helpful assistant."
USER = "Which letter comes first in the alphabet?\nA. B\nB. A\nC. D\nD. C\nAnswer with the letter only."

inputs, prompt = qo.make_inputs(processor, SYSTEM, USER)
print(prompt)
r = qo.score_options(model, processor, inputs)
print("prob A-D:", [round(p, 3) for p in r["prob"]])
print("mass on letters:", round(r["mass_on_letters"], 3))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Which letter comes first in the alphabet?
A. B
B. A
C. D
D. C
Answer with the letter only.<|im_end|>
<|im_start|>assistant

prob A-D: [0.01, 0.99, 0.0, 0.0]
mass on letters: 0.998


Same input again must give exactly the same numbers:

In [4]:
r2 = qo.score_options(model, processor, inputs)
print("identical:", r["logprob"] == r2["logprob"])

identical: True


## 3. One silent video clip, 64 frames

Frames are read by Qwen's own `qwen-omni-utils` code (video only, no audio) from the focused window `[t_i, t_j]` of one **validation** question (real question and options; the answer key is not used).
Two sizes: Qwen's default (full 840×476 for our films) and a smaller fallback in case the default is too slow.

In [9]:
q = pd.read_parquet(ROOT / "data/processed/questions.parquet").set_index("question_id").loc["xjVS2"]  # validation split
video = ROOT / "data/raw/all_videos" / f"{q.movie_title}.mp4"
options = "\n".join(f"{L}. {q['option_' + L]}" for L in qo.LETTERS)
user_text = f"Transcript:\n{q.transcript_focused}\n\nQuestion: {q.question.strip()}\n{options}\nAnswer with the letter only."
print(q.movie_title, f"| clip {q.t_j - q.t_i:.0f} s\n")
print(user_text)

FATHER FIGURE | clip 31 s

Transcript:
Tell you what, just hit one in and then you can go.
You'll probably do better than I'm doing now.
What do you say?
Fine.

Question: What is the emotional progression of the man with glasses in this scene?
A. Excitement → Anger → Disapproval
B. Joy → Amusement → Confusion
C. Excitement → Disappointment → Realization
D. Joy → Sadness → Neutral
Answer with the letter only.


In [10]:
rows = []
for max_pixels in ["text only", None, 160 * 28 * 28]:
    t0 = time.time()
    if max_pixels == "text only":
        inputs, _ = qo.make_inputs(processor, SYSTEM, user_text)
        frame_size = None
    else:
        frames, fps = qo.sample_frames(video, q.t_i, q.t_j, 64, max_pixels)
        inputs, _ = qo.make_inputs(processor, SYSTEM, user_text, frames, fps)
        frame_size = tuple(frames.shape[2:])
    t1 = time.time()
    r = qo.score_options(model, processor, inputs)
    t2 = time.time()
    rows.append({"setting": max_pixels or "default", "frame_size": frame_size, "input_tokens": r["n_input_tokens"],
                 "prepare_s": round(t1 - t0, 1), "model_s": round(t2 - t1), "total_s": round(t2 - t0),
                 "mass_on_letters": round(r["mass_on_letters"], 3), "prob_A_D": [round(p, 3) for p in r["prob"]]})
    print(rows[-1])
pd.DataFrame(rows)

{'setting': 'text only', 'frame_size': None, 'input_tokens': 119, 'prepare_s': 0.0, 'model_s': 1, 'total_s': 1, 'mass_on_letters': 0.999, 'prob_A_D': [0.075, 0.181, 0.716, 0.028]}
{'setting': 'default', 'frame_size': (476, 840), 'input_tokens': 16441, 'prepare_s': 0.5, 'model_s': 263, 'total_s': 263, 'mass_on_letters': 1.0, 'prob_A_D': [0.06, 0.087, 0.828, 0.025]}
{'setting': 125440, 'frame_size': (252, 448), 'input_tokens': 4729, 'prepare_s': 0.3, 'model_s': 64, 'total_s': 64, 'mass_on_letters': 1.0, 'prob_A_D': [0.042, 0.1, 0.838, 0.02]}


,setting,frame_size,input_tokens,prepare_s,model_s,total_s,mass_on_letters,prob_A_D
0,text only,None,119,0.0,1,1,0.999,"[0.075, 0.181, 0.716, 0.028]"
1,default,"(476, 840)",16441,0.5,263,263,1.000,"[0.06, 0.087, 0.828, 0.025]"
2,125440,"(252, 448)",4729,0.3,64,64,1.000,"[0.042, 0.1, 0.838, 0.02]"


## 4. Was the answer right?

In [11]:
key = pd.read_parquet(ROOT / "data/processed/answer_keys.parquet").set_index("question_id").loc[q.name, "correct_answer_key"]
probs = dict(zip(qo.LETTERS, [round(p, 3) for p in r["prob"]]))
print("prob A-D (last run):", probs)
print("model answer:", max(probs, key=probs.get), "| correct:", key, "| prob on correct:", probs[key])

prob A-D (last run): {'A': 0.042, 'B': 0.1, 'C': 0.838, 'D': 0.02}
model answer: C | correct: C | prob on correct: 0.838
